이 자료는 위키독스 딥 러닝을 이용한 자연어 처리 입문의 허깅페이스 토크나이저 학습 자료입니다.  

링크 : https://wikidocs.net/99893

# 1. BERT의 워드피스 토크나이저(BertWordPieceTokenizer)

In [1]:
pip install tokenizers

  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 21.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 31.0 MB/s  0:00:00
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
Using cached h11-0.16.0-py3-none-any.whl (37 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [tokenizers]8 [huggingface-hub]
Note: you may need to restart the kernel to use updated packages.


In [2]:
import tokenizers
tokenizers.__version__

'0.22.1'

In [3]:
import pandas as pd
import urllib.request
from tokenizers import BertWordPieceTokenizer

In [4]:
urllib.request.urlretrieve("https://raw.githubusercontent.com/e9t/nsmc/master/ratings.txt", filename="data/ratings.txt")

('data/ratings.txt', <http.client.HTTPMessage at 0x7aac744cc620>)

In [5]:
naver_df = pd.read_table('data/ratings.txt')
naver_df.head()

,id,document,label
0,8112052,어릴때보고 지금다시봐도 재밌어요ㅋㅋ,1
1,8132799,"디자인을 배우는 학생으로, 외국디자이너와 그들이 일군 전통을 통해 발전해가는 문화산...",1
2,4655635,폴리스스토리 시리즈는 1부터 뉴까지 버릴께 하나도 없음.. 최고.,1
3,9251303,와.. 연기가 진짜 개쩔구나.. 지루할거라고 생각했는데 몰입해서 봤다.. 그래 이런...,1
4,10067386,안개 자욱한 밤하늘에 떠 있는 초승달 같은 영화.,1


In [6]:
naver_df = naver_df.dropna(how = 'any') # Null 값이 존재하는 행 제거
print(naver_df.isnull().values.any()) # Null 값이 존재하는지 확인

False


In [7]:
with open('data/naver_review.txt', 'w', encoding='utf8') as f:
    f.write('\n'.join(naver_df['document']))

In [8]:
tokenizer = BertWordPieceTokenizer(lowercase=False, strip_accents=False)

lowercase : True일 경우 토크나이저는 영어의 대문자와 소문자를 동일한 문자 취급.  
strip_accents : True일 경우 악센트 제거.  
 ex) é → e, ô → o  
wordpieces_prefix : 서브워드로 쪼개졌을 경우 뒤의 서브워드에는 ##를 부착하여 원래 단어에서 분리된 것임을 표시.  
  ex) 안녕하세요 -> [안녕, ##하세요]  

In [10]:
data_file = 'data/naver_review.txt'
vocab_size = 30000
limit_alphabet = 6000
min_frequency = 5

tokenizer.train(files=data_file,
                vocab_size=vocab_size,
                limit_alphabet=limit_alphabet,
                min_frequency=min_frequency,
                )

vocab_size : 단어 집합의 크기  
limit_alphabet : merge가 되지 않은 초기 토큰(character 단위)의 허용 제한 개수  
min_frequency : merge가 되기 위한 pair의 최소 등장 횟수


In [11]:
# vocab 저장
tokenizer.save_model('models/')

['models/vocab.txt']

In [12]:
# vocab 로드
df = pd.read_fwf('models/vocab.txt', header=None)
df

,0
0,[PAD]
1,[UNK]
2,[CLS]
3,[SEP]
4,[MASK]
...,...
29995,말라는
29996,말밖에는
29997,맘을
29998,맛도


In [13]:
encoded = tokenizer.encode('아 배고픈데 짜장면먹고싶다')
print('토큰화 결과 :',encoded.tokens)
print('정수 인코딩 :',encoded.ids)
print('디코딩 :',tokenizer.decode(encoded.ids))

토큰화 결과 : ['아', '배고', '##픈', '##데', '짜장면', '##먹고', '##싶다']
정수 인코딩 : [2111, 20631, 3848, 3264, 24681, 7873, 7378]
디코딩 : 아 배고픈데 짜장면먹고싶다


In [14]:
encoded = tokenizer.encode('커피 한잔의 여유를 즐기다')
print('토큰화 결과 :',encoded.tokens)
print('정수 인코딩 :',encoded.ids)
print('디코딩 :',tokenizer.decode(encoded.ids))

토큰화 결과 : ['커피', '한잔', '##의', '여유', '##를', '즐기', '##다']
정수 인코딩 : [12825, 25648, 3297, 12696, 3280, 10784, 3289]
디코딩 : 커피 한잔의 여유를 즐기다


# 2. 기타 토크나이저

In [15]:
from tokenizers import ByteLevelBPETokenizer, CharBPETokenizer, SentencePieceBPETokenizer
                            
tokenizer = SentencePieceBPETokenizer()
tokenizer.train('data/naver_review.txt', vocab_size=10000, min_frequency=5)

encoded = tokenizer.encode("이 영화는 정말 재미있습니다.")
print(encoded.tokens)




['▁이', '▁영화는', '▁정말', '▁재미있', '습니다.']
